<a href="https://colab.research.google.com/github/smkim0508/cos484-notes/blob/main/A2P2_Named_Entity_Recognition_LSTMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook for Programming in Problem 2


## Learning Objectives
In this problem, we will use [PyTorch](https://pytorch.org/) to implement long short-term memory (LSTM) for named entity recognition (NER).

Before you start you may want to test that you have a GPU available in this Colab notebook.

In [17]:
!nvidia-smi # you may need to try reconnecting to get a T4 gpu

Sat Mar  7 04:07:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             32W /   70W |     371MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Installing PyTorch and Other Packages

You will need to install [PyTorch](https://pytorch.org/). You should also test that PyTorch is installed correctly and you can use GPU with PyTorch.

Also install [scikit-learn](https://scikit-learn.org/stable/). You can use it for calculating evaluation metrics such as accuracy and F1 score.

In [18]:
# import torch, sklearn, and test that GPU is available for torch
import torch
import sklearn

if torch.cuda.is_available():
  print('Torch is using GPU')

Torch is using GPU


## Long Short Term Memory (LSTM)

### Data Loading (5 points)

We will use the same dataset for named entity recognition in the previous programming problem. First download the data and inspect the data.


https://princeton-nlp.github.io/cos484/assignments/a2/eng.train

https://princeton-nlp.github.io/cos484/assignments/a2/eng.val

Afterwards, you should implement the logic for reading and processing the data files (may look similar to what you did in the previous programming problem).
You should also implement a `eval_metrics` functions that takes in a list of ground truth tags and prediction tags, then calculate and output the accuracy, f1 score, and confuction matrix.

Each line corresponds to a word. Different sentences are separated by an additional line break. Take "EU NNP I-NP ORG" as an example. "EU" is a word. "NNP" and "I-NP" are tags for POS tagging and chunking, which we will ignore. "ORG" is the tag for NER, which is our prediction target. There are 5 possible values for the NER tag: ORG, PER, LOC, MISC, and O.



In [19]:
# download both train and valid data
import urllib.request

TRAIN_URL = "https://princeton-nlp.github.io/cos484/assignments/a2/eng.train"
VALID_URL = "https://princeton-nlp.github.io/cos484/assignments/a2/eng.val"

urllib.request.urlretrieve(TRAIN_URL, "eng.train")
urllib.request.urlretrieve(VALID_URL, "eng.val")

('eng.val', <http.client.HTTPMessage at 0x78342277b950>)

### Data Loading **(continued)**

Like before, we first implement the data loader. Unlike before, each data example is now a variable-length sentence, but each PyTorch processes input in batches/tensors, where each item in the batch must have the same length (this is analogous to having a matrix of size batch_size x sequence_length).
How can we pack multiple sentences with different lengths into the same batch? One possible solution is to pad them to the same length using a special token.

You should implement a Dataset class that can form batches from multiple sentences by padding the sequences to the same length. You logic should enable the LSTM model to distinguish between regular and padded tokens (this will be important for loss calculations later).

Optionally, see [here](https://docs.pytorch.org/docs/stable/data.html#loading-batched-and-non-batched-data) for how batching works in PyTorch.

Finally, you should have a function for creating [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) out of the loaded training and evaluation datasets. You will use this for the training loop later.

In [20]:
# dataloader and dataset utils
import os
import re
import urllib.request
from typing import Optional

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

class NERSequenceData():
    """
    Reads NER data into sentences and builds id maps from train.

    File format per line: (WORD POS CHUNK NER), parsed to WORD and NER only.
    - sentences are separated by blank lines.
    """

    # handle padding and unseen words
    PAD = "<PAD>"
    UNK = "<UNK>"

    def __init__(self, normalize_digits: bool = True):
        self.normalize_digits = normalize_digits # optionally preprocess any digits to 0 to reduce vocab, as in problem 1
        self._digit_re = re.compile(r"\d")

        self.word_to_id: dict[str, int] = {self.PAD: 0, self.UNK: 1}
        self.tag_to_id: dict[str, int] = {self.PAD: 0}

        self.train: list[tuple[list[str], list[str]]] = []
        self.valid: list[tuple[list[str], list[str]]] = []

    # helper to load data into internal state
    # NOTE: using is_train bool to determine train vs valid
    def load(self, path: str, *, is_train: bool) -> None:
        sents = self._read_file(path)
        if is_train:
            self.train = sents
        else:
            self.valid = sents

    # maps word and tag to id internally
    def build_id_maps(self) -> None:
        if not self.train:
            raise ValueError("Load training data first (split='train') before building id maps.")

        for words, tags in self.train:
            for w in words:
                if w not in self.word_to_id:
                    self.word_to_id[w] = len(self.word_to_id)
            for t in tags:
                if t not in self.tag_to_id:
                    self.tag_to_id[t] = len(self.tag_to_id)

    # encode input words using map
    def encode(self, words: list[str], tags: Optional[list[str]] = None):
        w = [self.word_to_id.get(x, self.word_to_id[self.UNK]) for x in words]
        if tags is None:
            return w
        y = [self.tag_to_id[t] for t in tags]
        return w, y

    # helper to parse file
    def _read_file(self, path: str) -> list[tuple[list[str], list[str]]]:
        sentences: list[tuple[list[str], list[str]]] = []
        cur_w: list[str] = []
        cur_t: list[str] = []

        def flush():
            nonlocal cur_w, cur_t
            if cur_w:
                sentences.append((cur_w, cur_t))
                cur_w, cur_t = [], []

        with open(path, "r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()
                if not line:
                    flush()
                    continue
                if "DOCSTART" in line:
                    continue

                parts = line.split()
                word = parts[0]
                tag = parts[-1]

                if self.normalize_digits:
                    word = self._digit_re.sub("0", word)

                cur_w.append(word)
                cur_t.append(tag)

        flush()
        return sentences

class NERDataset(Dataset):
    """
    Helper class for the dataset.

    Each item of the form: (word_ids: list[int], tag_ids: list[int])
    Padding happens in collate_fn.
    """
    def __init__(self, examples: list[tuple[list[str], list[str]]], data: NERSequenceData):
        self.examples = examples
        self.data = data
        self.pad_word_id = data.word_to_id[data.PAD]
        self.pad_tag_id = data.tag_to_id[data.PAD]

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        words, tags = self.examples[idx]
        return self.data.encode(words, tags)

    def collate_fn(self, batch):
        # batch: list[(w_ids, t_ids)] where each is a python list[int]
        max_len = max(len(w) for w, _ in batch)
        B = len(batch)

        x = torch.full((B, max_len), self.pad_word_id, dtype=torch.long)
        y = torch.full((B, max_len), self.pad_tag_id, dtype=torch.long)
        mask = torch.zeros((B, max_len), dtype=torch.bool) # need a mask for real tokens

        lengths = torch.zeros((B,), dtype=torch.long)

        for i, (w, t) in enumerate(batch):
            n = len(w)
            lengths[i] = n
            x[i, :n] = torch.tensor(w, dtype=torch.long)
            y[i, :n] = torch.tensor(t, dtype=torch.long)
            mask[i, :n] = True

        return x, y, mask, lengths

# builds data loader using above classes
# NOTE: wraps torch's DataLoader class
def make_dataloaders(
    train_path: str,
    valid_path: str,
    *,
    batch_size: int = 32,
    shuffle_train: bool = True,
    normalize_digits: bool = True
):
    data = NERSequenceData(normalize_digits=normalize_digits)
    # NOTE: uses is_train bool to set train vs val sets
    data.load(train_path, is_train=True)
    data.load(valid_path, is_train=False)
    data.build_id_maps()

    train_ds = NERDataset(data.train, data)
    valid_ds = NERDataset(data.valid, data)

    train_dl = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle_train,
        collate_fn=train_ds.collate_fn,
    )
    valid_dl = DataLoader(
        valid_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=valid_ds.collate_fn,
    )

    return data, train_dl, valid_dl

# evaluation metrics and utils, to pipeline results in the future
def eval_metrics(
    y_true: list[int],
    y_pred: list[int],
    *,
    labels: list[int],
) -> dict:
    acc = accuracy_score(y_true, y_pred)
    f1_each = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return {"accuracy": acc, "f1_each": f1_each, "f1_macro": f1_macro, "confusion_matrix": cm}

def flatten_predictions(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    mask: torch.Tensor,
    pad_id: int,
):
    # y_true/y_pred: (B, L), mask: (B, L) True for non-pad tokens
    # returns python lists of ints (no pads)
    keep = mask & (y_true != pad_id)
    t = y_true[keep].tolist()
    p = y_pred[keep].tolist()
    return t, p

## Long Short-term Memory (LSTM)

Now we implement an one-layer LSTM for the same task as HMM.

### Implement the Model **(8 points)**

Next, implement LSTM for predicting NER tags from input words. [nn.LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html#torch.nn.LSTM) is definitely useful. Further, it is tricky to handle sentences in the same batch with different lengths. Please read the PyTorch documentation in detail!

Inside of your LSTM class, you may only use nn.Embedding, nn.Linear, nn.LSTM, and any nn.functional. You may be able to support creating LSM with the following inputs:
- `words_vocab` and `tags_vocab`: the vocabulary mapping from words and tags to ids
- `d_emb`: dimension size of the embedding
- `d_hidden`: dimension size of the hidden states
- `bidirection`: boolean indicating whether the LSTM is bidirectional or unidirectional.

The forward function should support taking a batch of data as inputs and outputs the model logits.

In [21]:
# the main LSTM model
import torch
import torch.nn as nn

class NERLSTM(nn.Module):
    """
    Main LSTM tagger for NER task.
    - Using nn modules

    - logits dim: B x L x C where C = num tags
    """

    def __init__(
        self,
        words_vocab: dict[str, int],
        tags_vocab: dict[str, int],
        *,
        d_emb: int = 100,
        d_hidden: int = 256,
        bidirection: bool = True,
        pad_word: str = "<PAD>",
        pad_tag: str = "<PAD>",
    ):
        super().__init__() # nn module init
        self.words_vocab = words_vocab
        self.tags_vocab = tags_vocab

        self.pad_word_id = words_vocab[pad_word]
        self.pad_tag_id = tags_vocab[pad_tag]

        self.V = len(words_vocab)
        self.C = len(tags_vocab)

        # if bi-directional model, increase fc layer dim to account for concat both direction hidden states
        # hidden dim multiplied by directions later
        self.directions = 2 if bidirection else 1

        self.emb = nn.Embedding(self.V, d_emb, padding_idx=self.pad_word_id)
        self.lstm = nn.LSTM(
            input_size=d_emb,
            hidden_size=d_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirection, # NOTE: nn.LSTM supports bi directional already
        )
        self.fc = nn.Linear(d_hidden * self.directions, self.C)

    # main forward pass
    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # x: (B, L), lengths: (B,)
        emb = self.emb(x) # tensor dim (B, L, d_emb)

        # pack -> run LSTM -> unpack back to (B, L, H*)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1)
        ) # (B, L, d_hidden*dir)

        logits = self.fc(out) # (B, L, C)
        return logits

### Training and Validation **(6 points)**

Write the functions for training and validating the LSTM model. When calculating the loss function, you only want to include values from valid positions (i.e., non-padded tokens). The `reduction` parameter in [F.cross_entropy](https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.cross_entropy) may be useful.
You may also use PyTorch optimizers, such as [SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html).

You should use GPU for training and evaluating your model for faster speeds.
When training and evaluating the models, you should keep track of the loss as well as the predictions for calculating metrics (using the function you wrote earlier).

In [22]:
import torch
import torch.nn.functional as F

# singular epoch training helper, wrapped in training method below
def run_epoch(
    model,
    dataloader,
    *,
    device: torch.device,
    optimizer=None, # eval mode if optimizer is None
    pad_tag_id: int,
    labels: list[int],
):
    # NOTE: check if optimizer exists to determine train vs eval mode
    train = optimizer is not None
    model.train(train)

    total_loss = 0.0
    total_tokens = 0

    all_true: list[int] = []
    all_pred: list[int] = []

    for x, y, mask, lengths in dataloader:
        x = x.to(device)
        y = y.to(device)
        mask = mask.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths) # (B, L, C)

        # token-level cross entropy, ignore PAD
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1),
            ignore_index=pad_tag_id,
            reduction="sum",
        )

        if train:
            # back prop if in training mode
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        # track the total loss
        total_loss += loss.item()

        with torch.no_grad():
            pred = logits.argmax(dim=-1) # (B, L)
            t, p = flatten_predictions(y, pred, mask, pad_tag_id)
            all_true.extend(t)
            all_pred.extend(p)
            total_tokens += len(t)

    avg_loss = (total_loss / total_tokens) if total_tokens else 0.0
    metrics = eval_metrics(all_true, all_pred, labels=labels)
    metrics["loss"] = avg_loss
    return metrics

# main training method
def train_lstm(
    model,
    train_dl,
    valid_dl,
    *,
    epochs: int = 5,
    lr: float = 0.1,
    device: Optional[torch.device] = None,
    pad_tag_id: int,
    labels: list[int],
):
    # use GPU when avail
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # alows computation in GPU using CUDA
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # runs training across multiple epochs
    for ep in range(1, epochs + 1):
        train_metrics = run_epoch(
            model,
            train_dl,
            device=device,
            optimizer=optimizer,
            pad_tag_id=pad_tag_id,
            labels=labels,
        )
        valid_metrics = run_epoch(
            model,
            valid_dl,
            device=device,
            optimizer=None,
            pad_tag_id=pad_tag_id,
            labels=labels,
        )

        # logs training progress
        print(
            f"epoch {ep} | "
            f"train loss {train_metrics['loss']:.4f} acc {train_metrics['accuracy']:.4f} f1 {train_metrics['f1_macro']:.4f} | "
            f"valid loss {valid_metrics['loss']:.4f} acc {valid_metrics['accuracy']:.4f} f1 {valid_metrics['f1_macro']:.4f}"
        )

    return model

Run the experiment with different hyperparameters (e.g., learning rate, batch size, dimension size, etc..).

Finally, use your best settings/hyperparameters to train a bidirectional LSTM, then re-run the same experiment with only unidirectional LSTM.

In [31]:
# hyperparameters
# NOTE: the current values are the best values found heuristically after iterations of experimenting
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.01
D_EMB = 100
D_HIDDEN = 256
NORMALIZE_DIGITS = True

TRAIN_PATH = "eng.train"
VALID_PATH = "eng.val"

# helper to run full experiment
def run_lstm_experiment(bidirectional: bool):
    data, train_dl, valid_dl = make_dataloaders(
        TRAIN_PATH,
        VALID_PATH,
        batch_size=BATCH_SIZE,
        shuffle_train=True,
        normalize_digits=NORMALIZE_DIGITS
    )
    # init model
    model = NERLSTM(
        data.word_to_id,
        data.tag_to_id,
        d_emb=D_EMB,
        d_hidden=D_HIDDEN,
        bidirection=bidirectional,
    )

    # encode
    pad_tag_id = data.tag_to_id[data.PAD]
    labels = [tid for tag, tid in data.tag_to_id.items() if tag != data.PAD]

    # train model
    model = train_lstm(
        model,
        train_dl,
        valid_dl,
        epochs=EPOCHS,
        lr=LR,
        pad_tag_id=pad_tag_id,
        labels=labels,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # get the final validation result w/ one epoch run and optimizer set to None to indicate eval
    final_valid = run_epoch(
        model,
        valid_dl,
        device=device,
        optimizer=None,
        pad_tag_id=pad_tag_id,
        labels=labels,
    )

    return model, data, final_valid

In [32]:
# runs both bi-directional and uni-directional experiments
print("Training bi-directional LSTM")
bi_model, bi_data, bi_valid = run_lstm_experiment(bidirectional=True)
print("bi-directional valid acc:", bi_valid["accuracy"], "f1:", bi_valid["f1_macro"])

print()
print("Training uni-directional LSTM")
uni_model, uni_data, uni_valid = run_lstm_experiment(bidirectional=False)
print("uni-directional valid acc:", uni_valid["accuracy"], "f1:", uni_valid["f1_macro"])

Training bi-directional LSTM
epoch 1 | train loss 0.9296 acc 0.8687 f1 0.5743 | valid loss 0.2486 acc 0.9295 f1 0.7408
epoch 2 | train loss 0.1535 acc 0.9545 f1 0.8398 | valid loss 0.2242 acc 0.9420 f1 0.7987
epoch 3 | train loss 0.0849 acc 0.9754 f1 0.9098 | valid loss 0.2871 acc 0.9436 f1 0.8095
epoch 4 | train loss 0.0463 acc 0.9872 f1 0.9507 | valid loss 0.2968 acc 0.9476 f1 0.8213
epoch 5 | train loss 0.0240 acc 0.9944 f1 0.9769 | valid loss 0.3233 acc 0.9473 f1 0.8242
epoch 6 | train loss 0.0126 acc 0.9976 f1 0.9906 | valid loss 0.3307 acc 0.9489 f1 0.8299
epoch 7 | train loss 0.0066 acc 0.9991 f1 0.9965 | valid loss 0.3227 acc 0.9512 f1 0.8358
epoch 8 | train loss 0.0038 acc 0.9996 f1 0.9986 | valid loss 0.3243 acc 0.9517 f1 0.8373
epoch 9 | train loss 0.0026 acc 0.9998 f1 0.9991 | valid loss 0.3390 acc 0.9513 f1 0.8365
epoch 10 | train loss 0.0018 acc 0.9999 f1 0.9996 | valid loss 0.3470 acc 0.9515 f1 0.8375
bi-directional valid acc: 0.9514525526626737 f1: 0.8375156355106661

T

### Questions **(6 points)**

(a) What is your strategy for choosing and validating hyperparameters? Which hyperparameters had/did not have the most impact on the final performance? You may support your answer by referring the experimental results above.

My strategy for choosing hyperparameters was starting out with something "standard", as popularly seen in modern NLP papers, then adjusting the values in small increments, one at a time and comparing the validation accuracy and f1 scores. I think it is important to only change one hyperparameter at a time, similiar to doing ablation study, so we can assess the impact of one parameter tuning at a time.

I found that the learning rate (LR) and hidden dimension size had the most impact to the experimental results, which makes sense since learning rate directly contributes to step size in gradient descent, and hidden dims represent the amount of trainable information or representable capacity. Of course, increasing the epoch to a reasonably high number also showed increasing progress, but I capped it at 10 to minimize computational overhead, and per-epoch gain slowed signicantly by 10 epochs already.

(b) How does bidirectional LSTMs compare to unidirectional LSTMs? Why?

I found that bi-directional LSTMs have higher validation accuracy than uni-directional LSTMs, which makes sense because they are able to better capture the whole context of the sequence when tagging it, whereas uni-directional LSTM only has past tokens to attend to and no information about future tokens (e.g. lack of context across entire sequence at each tagging step). Importantly, the model has learned future context during inference. This is also supported by a higher F1 score, indicating better recall and precision. However, a downside is that bi-directional LSTM is larger and more computationally costly/slower.

(c) Why are the masks necessary? How is it used in the loss and accuracy calculations? What if we did not use the masks?

Masking is necessary because we only want to train and predict on "real" tokens, i.e. the non-padded tokens. It is used in loss and accuracy by skipping evaluation on masked tokens, which is important to ensure our metrics are valid. Without masking step, the evaluation results could be very misleading, because it could lead to undesirable/unexpected behaviors trying to predict/fit to masked tokens (no real labels), and the accuracy could be misleadingly high due to good predictions on trivial masked tokens, which appear frequently throughout training data.



# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   
*   



**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)